# Exploración de imágenes Sentinel-2 sobre Posadas

**Fase 1 - exploración preliminar**

Este notebook explora la colección `COPERNICUS/S2_SR_HARMONIZED` sobre el área de Posadas durante la ventana de invierno seco (junio-agosto 2024), calcula un composite median, y visualiza NDBI y NDVI.

**Requiere**: variable de entorno `EE_PROJECT_ID` configurada (ver `.env.example`).

Fecha: 2026-04-22

In [ ]:
# Setup Earth Engine
import os
from pathlib import Path

import ee
from dotenv import load_dotenv
from IPython.display import Image, display

load_dotenv('../.env')

# Descomentar en la primera corrida para autenticarse:
# ee.Authenticate()

EE_PROJECT = os.getenv('EE_PROJECT_ID')
if not EE_PROJECT:
    raise EnvironmentError(
        'Configurá EE_PROJECT_ID en .env antes de correr este notebook.'
    )
ee.Initialize(project=EE_PROJECT)
print(f'Earth Engine inicializado con project = {EE_PROJECT}')

In [ ]:
# Cargar colección S2_SR_HARMONIZED y filtrar por Posadas junio-agosto 2024
BBOX_POSADAS = [-56.00, -27.50, -55.80, -27.30]  # oeste, sur, este, norte
AOI = ee.Geometry.Rectangle(BBOX_POSADAS)

col = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(AOI)
    .filterDate('2024-06-01', '2024-09-01')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
)
n_imagenes = col.size().getInfo()
print(f'Imágenes disponibles (nubes < 20%): {n_imagenes}')

In [ ]:
# Composite median y reflectancia promedio por banda
BANDAS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
composite = col.select(BANDAS).median().clip(AOI)

stats = composite.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=AOI,
    scale=30,
    maxPixels=1e9,
).getInfo()
print('Reflectancia promedio por banda (0-10000):')
for banda in BANDAS:
    valor = stats.get(banda)
    if valor is not None:
        print(f'  {banda}: {valor:.1f}')

In [ ]:
# Thumbnail RGB inline
vis_rgb = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000,
    'gamma': 1.3,
}
url_rgb = composite.getThumbURL({**vis_rgb, 'region': AOI, 'dimensions': 512})
print(f'URL thumbnail RGB: {url_rgb}')
display(Image(url=url_rgb))

In [ ]:
# NDBI = (B11 - B8) / (B11 + B8), NDVI = (B8 - B4) / (B8 + B4)
ndbi = composite.normalizedDifference(['B11', 'B8']).rename('NDBI')
ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')

vis_ndbi = {'min': -0.5, 'max': 0.5, 'palette': ['blue', 'white', 'red']}
vis_ndvi = {'min': -0.2, 'max': 0.8, 'palette': ['brown', 'yellow', 'green']}

url_ndbi = ndbi.getThumbURL({**vis_ndbi, 'region': AOI, 'dimensions': 512})
url_ndvi = ndvi.getThumbURL({**vis_ndvi, 'region': AOI, 'dimensions': 512})

print('NDBI (rojo = construido, azul = agua/vegetación):')
display(Image(url=url_ndbi))
print('NDVI (verde = vegetación, marrón = suelo/construido):')
display(Image(url=url_ndvi))

## Interpretación esperada

- **Composite median invierno seco**: debería tener cobertura completa y baja presencia de nubes.
- **NDBI alto (rojo)**: coincide con zonas urbanas densas del centro de Posadas y con zonas de expansión reciente (Itaembé Miní, Itaembé Guazú).
- **NDVI alto (verde)**: costa del Paraná con vegetación riparia, zonas rurales al sur del bbox.
- **Contraste NDBI vs. NDVI**: una máscara `NDBI > 0 AND NDVI < 0.3` aproxima edificado en un composite limpio.

**Nota**: este notebook requiere `EE_PROJECT_ID` configurado en `.env` y acceso autenticado a Earth Engine (`ee.Authenticate()` en la primera corrida). Las descargas efectivas a disco se hacen con `scripts/01_descarga_sentinel.py`.